In [3]:
from pathlib import Path
import re

import pandas as pd
from pandas import DataFrame

from config import PATHS
from helpers import rename_patient

In [4]:
# Paths
# on server:
# in_path = PATHS.per_model_comparison_table.pickle  # on server
# out_path = Path("~/thesis_data/tables/model_eval.tex").expanduser()

# local:
in_path = Path('~/thesis_files/statistical_results/.per_model.pkl').expanduser()  # local
out_path = Path('~/Developer/MastersThesis/src/tables/model_eval_metrics.tex').expanduser()

In [5]:
per_model = pd.read_pickle(in_path)
fake_mask = per_model.index.get_level_values('patient').str.contains('FAKE')
per_model = per_model[~fake_mask]
per_model

metric                 best_threshold            rel_tifw            \
split                           train      test     train      test   
patient       model                                                   
competition-1 CNN            0.730252  0.348021  0.150953  0.407381   
              ensemble       0.548969  0.420712  0.368110  0.494720   
competition-2 CNN            0.592629  0.503402  0.196632  0.330344   
              ensemble       0.604010  0.561291  0.323201  0.397423   
competition-3 CNN            0.779933  0.696605  0.142371  0.263264   
              ensemble       0.654698  0.622529  0.217113  0.285408   
U002-DE01-01  CNN            0.582112  0.634063  0.226472  0.165048   
              ensemble       0.567867  0.544376  0.266903  0.239180   
U002-DE01-03  CNN            0.780604  0.674600  0.107185  0.373027   
              ensemble       0.574479  0.525727  0.186428  0.426924   
U002-DE01-04  CNN            0.599625  0.559766  0.269078  0.472266   
              ensemble       0.526974  0.505274  0.430727  0.600760   
U002-DE01-05  CNN            0.756170  0.598670  0.065411  0.217563   
              ensemble       0.587826  0.586498  0.171846  0.189639   
U002-DE01-07  CNN            0.736176  0.701969  0.034486  0.183846   
              ensemble       0.545029  0.536581  0.282319  0.457661   
U002-DE01-12  CNN            0.666480  0.465285  0.072165  0.394819   
              ensemble       0.563852  0.561054  0.420331  0.482148   
U002-DE01-15  CNN            0.967000  0.868920  0.047427  0.345278   
              ensemble       0.561869  0.539433  0.315991  0.413938   
U002-DE01-16  CNN            0.812213  0.752847  0.130959  0.350571   
              ensemble       0.563557  0.559030  0.168770  0.215980   
U002-DE01-17  CNN            0.667585  0.528467  0.156727  0.358979   
              ensemble       0.609517  0.604358  0.345519  0.305847   

metric                 rel_szrs_pred           event_based_f1            \
split                          train      test          train      test   
patient       model                                                       
competition-1 CNN           0.794118  0.541667       0.820664  0.565998   
              ensemble      0.852941  0.500000       0.725961  0.502626   
competition-2 CNN           0.967742  0.863636       0.877928  0.754376   
              ensemble      0.838710  0.636364       0.749106  0.619009   
competition-3 CNN           0.945946  0.846154       0.899625  0.787663   
              ensemble      0.837838  0.961538       0.809431  0.819874   
U002-DE01-01  CNN           0.880342  0.794872       0.823486  0.814419   
              ensemble      0.803419  0.782051       0.766649  0.771290   
U002-DE01-03  CNN           0.852941  0.666667       0.872423  0.646211   
              ensemble      0.735294  0.444444       0.772455  0.500630   
U002-DE01-04  CNN           0.783784  0.560000       0.756430  0.543388   
              ensemble      0.729730  0.400000       0.639591  0.399620   
U002-DE01-05  CNN           0.800000  0.857143       0.862073  0.818088   
              ensemble      0.775000  0.785714       0.800696  0.797847   
U002-DE01-07  CNN           1.000000  0.666667       0.982454  0.733875   
              ensemble      0.857143  0.833333       0.781237  0.657059   
U002-DE01-12  CNN           0.777778  0.666667       0.846206  0.634438   
              ensemble      0.833333  0.750000       0.683732  0.612672   
U002-DE01-15  CNN           0.769231  0.700000       0.851141  0.676604   
              ensemble      0.846154  0.600000       0.756490  0.592949   
U002-DE01-16  CNN           1.000000  0.625000       0.929932  0.636981   
              ensemble      0.750000  0.750000       0.788529  0.766633   
U002-DE01-17  CNN           0.833333  0.777778       0.838274  0.702808   
              ensemble      0.833333  0.777778       0.733157  0.733590   

metric                 precision              recall    

In [6]:
t: DataFrame = per_model.copy()

t = t * 100 # put in percent
# Abbreviate patients
idx_df = t.index.to_frame(index=False)
idx_df['patient'] = idx_df['patient'].map(rename_patient)
t.index = pd.MultiIndex.from_frame(idx_df)

t = t.drop(columns=[
    # 'best_threshold',
    'precision',
    'recall',
], level='metric')

t = t.rename_axis(index={'patient': 'Patient', 'model': 'Model'},
                  columns={'metric': 'Metric:', 'split': 'Set:'})
t = t.rename(columns={
    'best_threshold': 'Thresh.',
    'rel_tifw': 'Rel. TIFW',
    'rel_szrs_pred': 'EB Sens.',
    'event_based_f1': 'EB Score',
    'precision': 'Precision',
    'recall': 'Recall',
    'roc_auc': 'ROC AUC',
    'p_hanley_mcneil': r'$p_{\mathrm{HM}}$',
    # This makes the table too wide
    # 'train': 'Train',
    # 'test': 'Test',
})

# Add min, max, mean rows
min_row = t.min(axis=0, numeric_only=True)
max_row = t.max(axis=0, numeric_only=True)
mean_row = t.mean(axis=0, numeric_only=True)

# Assign names to the index for these summary rows
min_row.name = ('Min', '')
max_row.name = ('Max', '')
mean_row.name = ('Mean', '')

# Append to the DataFrame
index_names = t.index.names
t = pd.concat([t, min_row.to_frame().T, max_row.to_frame().T, mean_row.to_frame().T])
t.index.names = index_names

t

Metric              Thresh.             Rel. TIFW               EB Sens.  \
Set                   train       test      train       test       train   
Patient Model                                                              
C01     CNN       73.025203  34.802124  15.095251  40.738093   79.411765   
        ensemble  54.896939  42.071170  36.811049  49.472005   85.294118   
C02     CNN       59.262931  50.340164  19.663169  33.034406   96.774194   
        ensemble  60.400975  56.129110  32.320063  39.742325   83.870968   
C03     CNN       77.993321  69.660503  14.237072  26.326385   94.594595   
        ensemble  65.469843  62.252885  21.711328  28.540753   83.783784   
U01     CNN       58.211178  63.406336  22.647192  16.504753   88.034188   
        ensemble  56.786734  54.437566  26.690257  23.917964   80.341880   
U03     CNN       78.060365  67.460024  10.718492  37.302676   85.294118   
        ensemble  57.447898  52.572662  18.642848  42.692402   73.529412   
U04     CNN       59.962547  55.976635  26.907818  47.226602   78.378378   
        ensemble  52.697384  50.527406  43.072734  60.075964   72.972973   
U05     CNN       75.617027  59.867007   6.541069  21.756295   80.000000   
        ensemble  58.782554  58.649826  17.184611  18.963940   77.500000   
U07     CNN       73.617613  70.196933   3.448612  18.384577  100.000000   
        ensemble  54.502922  53.658110  28.231856  45.766129   85.714286   
U12     CNN       66.648036  46.528545   7.216495  39.481903   77.777778   
        ensemble  56.385243  56.105387  42.033050  48.214833   83.333333   
U15     CNN       96.700019  86.892021   4.742679  34.527849   76.923077   
        ensemble  56.186914  53.943300  31.599121  41.393771   84.615385   
U16     CNN       81.221294  75.284684  13.095905  35.057072  100.000000   
        ensemble  56.355727  55.902994  16.876965  21.598015   75.000000   
U17     CNN       66.758549  52.846658  15.672725  35.897912   83.333333   
        ensemble  60.951704  60.435760  34.551862  30.584687   83.333333   
Min               52.697384  34.802124   3.448612  16.504753   72.972973   
Max               96.700019  86.892021  43.072734  60.075964  100.000000   
Mean              64.914288  57.914492  21.238009  34.883388   83.742121   

Metric                        EB Score               ROC AUC             \
Set                    test      train       test      train       test   
Patient Model                                                             
C01     CNN       54.166667  82.066443  56.599847  88.480453  61.068778   
        ensemble  50.000000  72.596100  50.262611  79.769776  59.267977   
C02     CNN       86.363636  87.792751  75.437569  89.322238  79.678225   
        ensemble  63.636364  74.910563  61.900950  77.012424  66.044048   
C03     CNN       84.615385  89.962529  78.766324  92.036783  85.317380   
        ensemble  96.153846  80.943071  81.987407  85.929523  85.490809   
U01     CNN       79.487179  82.348574  81.441930  86.320669  81.738358   
        ensemble  78.205128  76.664892  77.128975  80.456687  78.947994   
U03     CNN       66.666667  87.242276  64.621099  89.531787  56.961640   
        ensemble  44.444444  77.245481  50.062963  75.650955  50.186935   
U04     CNN       56.000000  75.643038  54.338843  78.293326  53.353268   
        ensemble  40.000000  63.959110  39.961982  65.555239  37.986387   
U05     CNN       85.714286  86.207317  81.808801  91.418314  81.346815   
        ensemble  78.571429  80.069576  79.784715  80.185793  76.279150   
U07     CNN       66.666667  98.245440  73.387531  91.610948  68.931663   
        ensemble  83.333333  78.123702  65.705911  73.059539  68.705851   
U12     CNN       66.666667  84.620551  63.443760  92.706698  61.366026   
        ensemble  75.000000  68.373241  61.267223  72.541324  66.486516   
U15     CNN       70.000000  85.114059  67.660409  87.931913  60.472662   
        ensemble  60.000000  75.649040  59.294925  75.739439  57.7697

In [7]:
latex = t.to_latex(
    float_format='%.0f',
    multirow=True,
    multicolumn=True,
    multicolumn_format='c',
)
# Remove superfluous clines
# remove cline before \\bottomrule
latex = re.sub(r'\\cline\{.*?\}\n\\bottomrule', r'\\bottomrule', latex)
# Replace cline before Min row with midrule
latex = re.sub(r'\\cline\{.*?\}\nMin', r'\\midrule\nMin', latex)

# Remove clines before Max and Mean rows
latex = re.sub(r'\\cline\{.*?\}\n(Max|Mean)', r'\g<1>', latex)

out_path.write_text(latex, encoding="utf-8")
print(latex)

\begin{tabular}{llrrrrrrrrrrrr}
\toprule
 & Metric & \multicolumn{2}{c}{Thresh.} & \multicolumn{2}{c}{Rel. TIFW} & \multicolumn{2}{c}{EB Sens.} & \multicolumn{2}{c}{EB Score} & \multicolumn{2}{c}{ROC AUC} & \multicolumn{2}{c}{$p_{\mathrm{HM}}$} \\
 & Set & train & test & train & test & train & test & train & test & train & test & train & test \\
Patient & Model &  &  &  &  &  &  &  &  &  &  &  &  \\
\midrule
\multirow[t]{2}{*}{C01} & CNN & 73 & 35 & 15 & 41 & 79 & 54 & 82 & 57 & 88 & 61 & 0 & 0 \\
 & ensemble & 55 & 42 & 37 & 49 & 85 & 50 & 73 & 50 & 80 & 59 & 0 & 0 \\
\cline{1-14}
\multirow[t]{2}{*}{C02} & CNN & 59 & 50 & 20 & 33 & 97 & 86 & 88 & 75 & 89 & 80 & 0 & 0 \\
 & ensemble & 60 & 56 & 32 & 40 & 84 & 64 & 75 & 62 & 77 & 66 & 0 & 0 \\
\cline{1-14}
\multirow[t]{2}{*}{C03} & CNN & 78 & 70 & 14 & 26 & 95 & 85 & 90 & 79 & 92 & 85 & 0 & 0 \\
 & ensemble & 65 & 62 & 22 & 29 & 84 & 96 & 81 & 82 & 86 & 85 & 0 & 0 \\
\cline{1-14}
\multirow[t]{2}{*}{U01} & CNN & 58 & 63 & 23 & 17 & 88 & 